# NovaRota: verificar Free Edition
Amostra isolada de capacidade. Usa schema e Volume proprios e dois registros CSV sinteticos. Preencha `catalog` com um catalogo existente no Catalog Explorer. Reexecutar preserva arquivos, tabela e checkpoint. Saidas ainda nao executadas neste arquivo.

In [ ]:
import json
import re
from datetime import datetime, timezone

dbutils.widgets.text('catalog', 'workspace')
dbutils.widgets.text('schema', 'novarota_smoke')
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
for name in (catalog, schema):
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', name):
        raise ValueError('Identificador de catalogo/schema invalido')

volume = f'{catalog}.{schema}.smoke_files'
root = f'/Volumes/{catalog}/{schema}/smoke_files'
table = f'{catalog}.{schema}.bronze_probe'

def setup():
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}')
    spark.sql(f'CREATE VOLUME IF NOT EXISTS {volume}')
    dbutils.fs.mkdirs(root + '/landing')

setup()
print(json.dumps({'started_at': datetime.now(timezone.utc).isoformat(), 'volume': root, 'table': table}))

In [ ]:
def publish(filename, content):
    existing = {entry.name for entry in dbutils.fs.ls(root + '/landing')}
    path = root + '/landing/' + filename
    if filename in existing:
        if dbutils.fs.head(path, 10000) != content:
            raise ValueError('Arquivo imutavel possui conteudo diferente: ' + path)
    else:
        dbutils.fs.put(path, content, overwrite=False)

def ingest():
    query = (spark.readStream.format('cloudFiles')
        .option('cloudFiles.format', 'csv')
        .option('header', 'true')
        .option('cloudFiles.schemaLocation', root + '/schemas/probe')
        .option('cloudFiles.schemaHints', 'id string, valor string')
        .option('cloudFiles.schemaEvolutionMode', 'rescue')
        .option('rescuedDataColumn', '_rescued_data')
        .load(root + '/landing')
        .writeStream.format('delta').outputMode('append')
        .option('checkpointLocation', root + '/checkpoints/probe')
        .trigger(availableNow=True).toTable(table))
    query.awaitTermination()
    if query.exception() is not None:
        raise RuntimeError(str(query.exception()))
    return spark.table(table).count()

publish('batch_001.csv', 'id,valor\nsmoke-001,100.00\n')
print({'stage': 'initial_or_resume', 'rows': ingest()})


In [ ]:
publish('batch_002.csv', 'id,valor,dispositivo\nsmoke-002,abc,mobile-sintetico\n')
assert ingest() == 2, 'Esperados dois registros no schema isolado'
result = spark.table(table)
assert result.where("id = 'smoke-002' AND valor = 'abc' AND get_json_object(_rescued_data, '$.dispositivo') = 'mobile-sintetico'").count() == 1, 'Campo novo ou valor bruto nao foi preservado'
assert ingest() == 2, 'Replay duplicou registros'
setup()
assert spark.table(table).count() == 2, 'Setup alterou tabela existente'
assert ingest() == 2, 'Setup alterou checkpoint existente'
display(spark.table(table))
print(json.dumps({'status': 'PASSED', 'checks': ['initial', 'increment', 'rescue_csv', 'invalid_lexical_value', 'replay', 'setup_preserves_state'], 'rows': 2, 'finished_at': datetime.now(timezone.utc).isoformat()}))